In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

from sklearn.feature_selection import SelectKBest, f_regression, mutual_info_regression, RFE
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score, KFold

N_FEATURES = 15


In [2]:
# Load and aggregate data
df = pd.read_csv(Path('../data/processed/Barcelona_3seasons_features.csv'))
metadata_cols = ['player_id', 'player_name', 'midfielder_type', 'match_id',
                 'team_id', 'team_name', 'season', 'computed_at', 'bypasses_per_halftime']
all_feature_cols = [c for c in df.columns if c not in metadata_cols]

# ── Drop only features that are NOT player-inherent ──────────────────────────
# defensive_shape_compactness: collective team metric, not individual
# zone14_touches: driven by positional role assignment, not spatial intelligence
DROP_FEATURES = ['defensive_shape_compactness', 'zone14_touches']

feature_cols = [c for c in all_feature_cols if c not in DROP_FEATURES]
print(f'Dropped {len(DROP_FEATURES)} non-player features: {DROP_FEATURES}')
print(f'Remaining: {len(feature_cols)} features (player-inherent spatial + skill + opponent context)')

# ── Aggregation rules ────────────────────────────────────────────────────────
MATCH_LEVEL_PREFIXES = ('opp_', 'score_diff')
MEAN_KEYWORDS = ('rate', 'position', 'index', '_x', '_y',
                 'variance', 'coverage', 'compactness', 'avg_', 'average_')

def _agg_rule(col):
    if any(col.startswith(p) for p in MATCH_LEVEL_PREFIXES):
        return 'first'   # match-level constant — same for all 3 players
    if any(k in col for k in MEAN_KEYWORDS):
        return 'mean'    # rate/position/spatial — average across players
    return 'sum'         # count — sum across players

agg_dict = {col: 'first' for col in ['team_id', 'team_name', 'season', 'bypasses_per_halftime']}
agg_dict.update({col: _agg_rule(col) for col in feature_cols})
match_df = df.groupby('match_id').agg(agg_dict).reset_index()

# Sanity checks
cov_x = match_df['midfield_zone_coverage_x']
print(f'midfield_zone_coverage_x range: {cov_x.min():.1f} - {cov_x.max():.1f}  (should be ~30-80)')
print(f'opp_ppda mean: {match_df["opp_ppda"].mean():.2f}')

X = match_df[[c for c in match_df.columns
              if c not in ['match_id','team_id','team_name','season','bypasses_per_halftime']]].copy()
y = match_df['bypasses_per_halftime'].copy()
print(f'X: {X.shape}, y: {y.shape}')
opp_cols = [c for c in X.columns if c.startswith('opp_') or c.startswith('score_')]
print(f'Opponent features ({len(opp_cols)}): {opp_cols}')


Dropped 2 non-player features: ['defensive_shape_compactness', 'zone14_touches']
Remaining: 80 features (player-inherent spatial + skill + opponent context)
midfield_zone_coverage_x range: 32.5 - 72.8  (should be ~30-80)
opp_ppda mean: 7.05
X: (204, 80), y: (204,)
Opponent features (8): ['opp_formation', 'opp_long_ball_rate', 'opp_avg_pass_length', 'opp_direct_play_index', 'opp_pass_forward_rate', 'opp_ppda', 'opp_press_actions', 'score_diff_start_of_half']


In [3]:
# Clean and scale
X_clean = X.drop(columns=X.isna().sum()[X.isna().sum() / len(X) > 0.5].index).fillna(X.median())
X_scaled = pd.DataFrame(StandardScaler().fit_transform(X_clean), columns=X_clean.columns, index=X_clean.index)


In [4]:
# Feature selection: Select top 5 features
X_filled = X_scaled.fillna(X_scaled.median())

# 1. Correlation
correlations = {col: X_scaled[col].corr(y) for col in X_scaled.columns}

# 2. Univariate (F-test + Mutual Information)
selector_f = SelectKBest(score_func=f_regression, k='all')
selector_f.fit(X_filled, y)
f_scores = pd.Series(selector_f.scores_, index=X_filled.columns)

selector_mi = SelectKBest(score_func=mutual_info_regression, k='all')
selector_mi.fit(X_filled, y)
mi_scores = pd.Series(selector_mi.scores_, index=X_filled.columns)

# Combine scores
feature_scores = pd.DataFrame({
    'F_Score': f_scores,
    'MI_Score': mi_scores,
    'Correlation': [correlations.get(f, 0) for f in X_filled.columns]
}, index=X_filled.columns)

feature_scores['F_Norm'] = (feature_scores['F_Score'] - feature_scores['F_Score'].min()) / \
                          (feature_scores['F_Score'].max() - feature_scores['F_Score'].min() + 1e-10)
feature_scores['MI_Norm'] = (feature_scores['MI_Score'] - feature_scores['MI_Score'].min()) / \
                           (feature_scores['MI_Score'].max() - feature_scores['MI_Score'].min() + 1e-10)
feature_scores['Corr_Norm'] = abs(feature_scores['Correlation'])
feature_scores['Combined_Score'] = (feature_scores['F_Norm'] + feature_scores['MI_Norm'] + 
                                    feature_scores['Corr_Norm']) / 3

feature_scores = feature_scores.sort_values('Combined_Score', ascending=False)
selected_univariate = feature_scores.head(N_FEATURES).index.tolist()

# 3. Random Forest
rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_filled, y)
rf_importance = pd.Series(rf.feature_importances_, index=X_filled.columns).sort_values(ascending=False)
selected_rf = rf_importance.head(N_FEATURES).index.tolist()

# 4. RFE
rfe = RFE(estimator=RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1), 
          n_features_to_select=N_FEATURES, step=1)
rfe.fit(X_filled, y)
selected_rfe = X_filled.columns[rfe.support_].tolist()


In [5]:
# Combine results and select final top 5
all_selections = {'Univariate': set(selected_univariate), 'RF': set(selected_rf), 'RFE': set(selected_rfe)}
selection_counts = {f: sum(1 for fset in all_selections.values() if f in fset) for f in X_scaled.columns}

final_scores = {}
for f in X_scaled.columns:
    count = selection_counts.get(f, 0)
    uni_score = feature_scores.loc[f, 'Combined_Score'] if f in feature_scores.index else 0
    rf_norm = rf_importance[f] / rf_importance.max() if rf_importance.max() > 0 else 0
    final_scores[f] = (count * 0.4) + (uni_score * 0.3) + (rf_norm * 0.3)

final_selected_features = pd.Series(final_scores).sort_values(ascending=False).head(N_FEATURES).index.tolist()
print(f"Selected {N_FEATURES} features:")
for i, f in enumerate(final_selected_features, 1):
    print(f"  {i}. {f}")




Selected 15 features:
  1. midfield_zone_coverage_x
  2. possessions_involved
  3. average_position_x
  4. penalty_area_deliveries
  5. avg_defensive_x_on_deep_opp
  6. interceptions
  7. opp_ppda
  8. bypass_channel_defensive_actions
  9. opp_avg_pass_length
  10. final_third_entries_by_pass
  11. width_variance
  12. tempo_index
  13. average_position_y
  14. under_pressure_pass_share
  15. weak_foot_pass_share


In [6]:
# Evaluate
kf = KFold(n_splits=5, shuffle=True, random_state=42)
rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
X_sel = X_scaled[final_selected_features]

scores_all = cross_val_score(rf, X_scaled, y, cv=kf, scoring='neg_mean_squared_error', n_jobs=-1)
r2_all = cross_val_score(rf, X_scaled, y, cv=kf, scoring='r2', n_jobs=-1)
scores_sel = cross_val_score(rf, X_sel, y, cv=kf, scoring='neg_mean_squared_error', n_jobs=-1)
r2_sel = cross_val_score(rf, X_sel, y, cv=kf, scoring='r2', n_jobs=-1)

print(f"All ({X_scaled.shape[1]}): MSE={-scores_all.mean():.2f}, R²={r2_all.mean():.4f}")
print(f"Selected {N_FEATURES}: MSE={-scores_sel.mean():.2f}, R²={r2_sel.mean():.4f}")




All (79): MSE=7.59, R²=0.0007
Selected 15: MSE=7.49, R²=0.0126


In [7]:
# Summary
print(f"{len(X_scaled)} samples, {len(X_scaled.columns)} features → {N_FEATURES} selected")
print(f"MSE={-scores_sel.mean():.2f}, R²={r2_sel.mean():.4f}")


204 samples, 79 features → 15 selected
MSE=7.49, R²=0.0126


In [8]:
# Export final selected features and create dataset
export_df = pd.DataFrame({'feature_name': final_selected_features})
export_path = Path("../data/processed/Barcelona_2014_2015_selected_features_final.csv")
export_df.to_csv(export_path, index=False)
with open(Path("../data/processed/Barcelona_2014_2015_selected_features_list.txt"), 'w') as f:
    for feat in final_selected_features:
        f.write(f"{feat}\n")
print(f"Exported {len(final_selected_features)} features")

# Create final dataset with selected features
dataset_cols = ['match_id', 'team_id', 'team_name', 'season'] + final_selected_features + ['bypasses_per_halftime']
final_dataset = match_df[dataset_cols].copy()
dataset_path = Path("../data/processed/Barcelona_2014_2015_selected_features.csv")
final_dataset.to_csv(dataset_path, index=False)
print(f"Dataset saved: {dataset_path} ({final_dataset.shape[0]} rows, {final_dataset.shape[1]} cols)")

# Create scaled version
scaled_dataset = match_df[['match_id', 'team_id', 'team_name', 'season']].copy()
for feat in final_selected_features:
    scaled_dataset[feat] = X_scaled[feat].values
scaled_dataset['bypasses_per_halftime'] = match_df['bypasses_per_halftime'].values
scaled_path = Path("../data/processed/Barcelona_2014_2015_selected_features_scaled.csv")
scaled_dataset.to_csv(scaled_path, index=False)
print(f"Scaled dataset saved: {scaled_path} ({scaled_dataset.shape[0]} rows, {scaled_dataset.shape[1]} cols)")


Exported 15 features
Dataset saved: ../data/processed/Barcelona_2014_2015_selected_features.csv (204 rows, 20 cols)
Scaled dataset saved: ../data/processed/Barcelona_2014_2015_selected_features_scaled.csv (204 rows, 20 cols)


In [9]:
# Export: 10 selected features per player per match-half (individual level)
player_metadata_cols = ['player_id', 'player_name', 'midfielder_type', 'match_id',
                        'team_id', 'team_name', 'season', 'bypasses_per_halftime']

# Only keep features that exist in the raw individual df (some may have been dropped during cleaning)
available_features = [f for f in final_selected_features if f in df.columns]
missing_features = [f for f in final_selected_features if f not in df.columns]
if missing_features:
    print(f"⚠️  Features not in raw df (dropped during cleaning): {missing_features}")

player_cols = player_metadata_cols + available_features
player_df = df[player_cols].copy()

player_path = Path("../data/processed/Barcelona_2014_2015_player_selected_features.csv")
player_df.to_csv(player_path, index=False)

print(f"✓ Individual player dataset saved: {player_path}")
print(f"  Shape: {player_df.shape}  ({player_df['player_id'].nunique()} unique players, "
      f"{player_df['match_id'].nunique()} match-halves)")
print(f"\nColumns: {player_df.columns.tolist()}")
print(f"\nSample (first 5 rows):")
print(player_df.head().to_string(index=False))


✓ Individual player dataset saved: ../data/processed/Barcelona_2014_2015_player_selected_features.csv
  Shape: (614, 23)  (10 unique players, 204 match-halves)

Columns: ['player_id', 'player_name', 'midfielder_type', 'match_id', 'team_id', 'team_name', 'season', 'bypasses_per_halftime', 'midfield_zone_coverage_x', 'possessions_involved', 'average_position_x', 'penalty_area_deliveries', 'avg_defensive_x_on_deep_opp', 'interceptions', 'opp_ppda', 'bypass_channel_defensive_actions', 'opp_avg_pass_length', 'final_third_entries_by_pass', 'width_variance', 'tempo_index', 'average_position_y', 'under_pressure_pass_share', 'weak_foot_pass_share']

Sample (first 5 rows):
 player_id                    player_name  midfielder_type  match_id  team_id team_name    season  bypasses_per_halftime  midfield_zone_coverage_x  possessions_involved  average_position_x  penalty_area_deliveries  avg_defensive_x_on_deep_opp  interceptions  opp_ppda  bypass_channel_defensive_actions  opp_avg_pass_length  fina

In [10]:
# Player summary: appearances in midfield across matches
player_df['base_match_id'] = player_df['match_id'].str.replace(r'_P\d+$', '', regex=True)

summary = (
    player_df.groupby(['player_id', 'player_name', 'midfielder_type'])
    .agg(
        match_halves=('match_id', 'count'),
        matches_played=('base_match_id', 'nunique'),
    )
    .reset_index()
    .sort_values('matches_played', ascending=False)
    .reset_index(drop=True)
)

# Map midfielder_type code to readable label
type_labels = {
    0: 'Left Midfield', 1: 'Right Midfield', 2: 'Center Midfield',
    3: 'Left Attacking Mid', 4: 'Center Defensive Mid', 5: 'Right Attacking Mid',
    6: 'Left Defensive Mid', 7: 'Right Defensive Mid'
}
summary['position'] = summary['midfielder_type'].map(type_labels).fillna('Unknown')

print(f"Player Midfield Appearance Summary  ({summary.shape[0]} players)\n")
print(f"{'#':<4} {'Player':<30} {'Position':<25} {'Matches':>8} {'Halves':>8}")
print("-" * 78)
for rank, row in summary.iterrows():
    print(f"{rank+1:<4} {row['player_name']:<30} {row['position']:<25} {row['matches_played']:>8} {row['match_halves']:>8}")

# Save
summary_path = Path("../data/processed/Barcelona_2014_2015_player_midfield_summary.csv")
summary[['player_id', 'player_name', 'position', 'matches_played', 'match_halves']].to_csv(summary_path, index=False)
print(f"\n✓ Summary saved to: {summary_path}")


Player Midfield Appearance Summary  (15 players)

#    Player                         Position                   Matches   Halves
------------------------------------------------------------------------------
1    Sergio Busquets i Burgos       Left Midfield                   74      148
2    Xavier Hernández Creus         Right Midfield                  61      122
3    Andrés Iniesta Luján           Right Midfield                  51      102
4    Francesc Fàbregas i Soler      Right Midfield                  40       79
5    Ivan Rakitić                   Right Midfield                  24       48
6    Alexandre Dimitri Song-Billong Left Midfield                   16       32
7    Rafael Alcântara do Nascimento Right Midfield                  12       24
8    Thiago Alcântara do Nascimento Right Midfield                  11       22
9    Javier Alejandro Mascherano    Left Midfield                    8       16
10   Sergi Roberto Carnicer         Left Midfield                    3 